In [ ]:
import os, glob, json
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from google.colab import drive
drive.mount("/content/drive")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

Mounted at /content/drive
device: cuda


In [ ]:
hits = glob.glob("/content/drive/MyDrive/**/data/passages.jsonl", recursive=True)
print("found:", hits)
PASSAGES = hits[0]                                   # change the index if several copies are listed
ROOT = PASSAGES.rsplit("/data/", 1)[0]
out_dir = lambda ds: f"{ROOT}/step1" if ds == "nq" else f"{ROOT}/step1_{ds}"


found: ['/content/drive/MyDrive/rag_trust/data/passages.jsonl']


In [ ]:
df = pd.DataFrame([json.loads(l) for l in open(PASSAGES)])     # keeps ids as strings
df["retrieval_score"] = pd.to_numeric(df["retrieval_score"])
g = df[df["context_id"].notna()].copy()               # DistilBERT-only rows have no group
g = g.sort_values(["context_id", "rank"]).reset_index(drop=True)
G = len(g) // 5
assert len(g) % 5 == 0 and (g.groupby("context_id").size() == 5).all(), "every group must have 5 rows"
assert (g["rank"].values.reshape(G, 5) == np.arange(1, 6)).all()
print(len(df), "rows;", len(g), "in groups;", G, "groups")


32252 rows; 28650 in groups; 5730 groups


In [ ]:
# Encoding convention (same as Step 1): clean-derived passages = title + " " + text, poison and injection = text only
DOCS = {ds: json.load(open(f"{out_dir(ds)}/docs.json")) for ds in g["dataset"].unique()}
def index_text(ds, source, pid, passage):
    if source == "clean":
        title = DOCS[ds][pid.split(":doc:", 1)[1]]["title"]
        return f"{title} {passage}" if title else passage
    return passage
g["itext"] = [index_text(*a) for a in zip(g.dataset, g.source, g.passage_id, g.passage)]

In [ ]:
CT, CE = f"{ROOT}/data/emb_texts.json", f"{ROOT}/data/emb.npy"
cache = dict(zip(json.load(open(CT)), np.load(CE))) if os.path.exists(CT) else {}
need = sorted(set(g["itext"]) | set(g["question"]), key=len)
todo = [t for t in need if t not in cache]
print(len(need), "unique texts;", len(todo), "to embed")


6315 unique texts; 593 to embed


In [ ]:
if todo:
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained("facebook/contriever")
    enc = AutoModel.from_pretrained("facebook/contriever").to(device).eval()   # pooler warning is expected
    @torch.no_grad()
    def encode(texts, bs=64):
        out = []
        for i in range(0, len(texts), bs):
            b = tok(texts[i:i + bs], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            h = enc(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            out.append(((h * m).sum(1) / m.sum(1)).float().cpu())     # masked mean pooling, no normalisation
        return torch.cat(out).numpy()
    new = encode(todo)
    cache.update(zip(todo, new))
    json.dump(list(cache), open(CT, "w")); np.save(CE, np.stack(list(cache.values())))

index = {t: i for i, t in enumerate(need)}
E = np.stack([cache[t] for t in need]).astype(np.float32)
EN = E / np.linalg.norm(E, axis=1, keepdims=True)            # L2-normalised, for the centroid feature
ip, iq = g["itext"].map(index).values, g["question"].map(index).values

def centroid_dist(X):
    """X: (groups, 5, D) L2-normalised. Cosine distance of each passage from the mean of the OTHER four."""
    others = (X.sum(1, keepdims=True) - X) / (X.shape[1] - 1)
    return 1 - (X * others).sum(-1) / np.linalg.norm(others, axis=-1)

# unit test: the centroid really excludes the passage itself
rng = np.random.default_rng(0)
X = rng.normal(size=(5, 16)); X /= np.linalg.norm(X, axis=1, keepdims=True)
for i in range(5):
    o = np.delete(X, i, axis=0).mean(0)
    assert abs(centroid_dist(X[None])[0, i] - (1 - X[i] @ o / np.linalg.norm(o))) < 1e-6
print("centroid unit test passed")

# features
f1 = np.empty(len(g), np.float32)                              # raw dot product, as in Step 1
for s in range(0, len(g), 20000):
    sl = slice(s, s + 20000); f1[sl] = (E[ip[sl]] * E[iq[sl]]).sum(1)
S = f1.reshape(G, 5)
f2 = (S - S.mean(1, keepdims=True)) / np.maximum(S.std(1, keepdims=True), 1e-6)    # z-score within the group
ipg = ip.reshape(G, 5); f3 = np.empty((G, 5), np.float32)
for s in range(0, G, 4000):
    f3[s:s + 4000] = centroid_dist(EN[ipg[s:s + 4000]])
g["f1"], g["f2"], g["f3"] = f1, f2.reshape(-1), f3.reshape(-1)

keep = ["dataset", "qid", "split", "context_id", "n_poison", "group_variant", "variant", "source",
        "poison_idx", "rank", "passage_id", "label", "retrieval_score", "f1", "f2", "f3"]
g[keep].to_parquet(f"{ROOT}/data/features.parquet", index=False)
print("saved features.parquet:", len(g), "rows")


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

centroid unit test passed
saved features.parquet: 28650 rows


In [ ]:
# ---- checks ----
o = g[g["variant"] == "original"]
d = (o["f1"] - o["retrieval_score"]).abs()
print("\nrecomputed vs stored score, by source:\n", d.groupby(o["source"]).agg(["max", "mean"]).round(4))

n0 = g[(g["group_variant"] == "original") & (g["n_poison"] == 0)]
piv = n0.pivot(index="context_id", columns="rank", values="f1")
print("n=0 groups: recomputed rank1 > rank5:", round(float((piv[1] > piv[5]).mean()), 3),
      "| strictly decreasing:", round(float((piv.diff(axis=1).iloc[:, 1:] < 0).all(axis=1).mean()), 3))

t = g[(g["split"] == "train") & (g["group_variant"] == "original") & (g["n_poison"] == 1)]
print("\nTRAIN only, original n=1 groups, single-feature AUROC (above 0.5 = higher means poisoned):")
for f in ("f1", "f2", "f3"):
    print(" ", f, round(roc_auc_score(t["label"], t[f]), 3))


recomputed vs stored score, by source:
         max  mean
source           
clean   0.0   0.0
poison  0.0   0.0
n=0 groups: recomputed rank1 > rank5: 1.0 | strictly decreasing: 0.997

TRAIN only, original n=1 groups, single-feature AUROC (above 0.5 = higher means poisoned):
  f1 0.985
  f2 0.993
  f3 0.458


In [ ]:
  print(len(o), "rows compared | max abs diff:", f"{d.max():.2e}", "| by source:", d.groupby(o['source']).max().map('{:.2e}'.format).to_dict())

20896 rows compared | max abs diff: 1.43e-06 | by source: {'clean': '9.54e-07', 'poison': '1.43e-06'}
